In [24]:
import os
import json
import pandas as pd
from openai import OpenAI
import dotenv

dotenv.load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

POSTS_PATH = "../../data/hannah_raw_posts.csv"
CHECKPOINT_PATH = "../../data/sentence_pool_checkpoint.json"
OUTPUT_PATH = "../../data/sentence_pool.csv"

# Stage 1: Sentence Extraction

In [25]:
SYSTEM_PROMPT = """\
You are extracting sentences from Reddit posts for a research project.
You are building a sentence pool for a fictional character called Hannah — a 16-year-old
girl in secondary school.

For each post, extract sentences that authentically describe experiences matching
one of these 6 sections. Return ONLY a JSON object with a "sentences" key.

SECTIONS:
- family_father: feelings or experiences relating to an absent or volatile father figure
- family_mother: feelings or experiences relating to an emotionally unavailable mother
- sa: sexual abuse or assault experiences, grooming, self-blame around it
- bullying: social exclusion, being targeted at school, peer cruelty
- self_harm: cutting or self-injury
- suicidal_ideation: passive thoughts of not wanting to exist, feeling like a burden

EXTRACT near-verbatim sentences. Preserve spelling errors, sentence fragments,
run-ons, and lowercase — do not correct the writing.

MINIMAL EDITS ONLY — change nothing except:
- Age: change to 16 if stated differently
- School level: university or college → secondary school or year 11
- Self-harm method: if burning or other method → change to cutting
- SA perpetrator: if described as a parent → change to "an older boy I knew"
- Specific suicide plan, date, or method → soften to passive ideation only

DISCARD sentences about:
- Anything that cannot be fixed with the minimal edits above

Return format:
{"sentences": [
  {"section": "bullying", "original": "...", "extracted": "..."},
  ...
]}

Return {"sentences": []} if no sentences are extractable.
"""

In [26]:
def extract_sentences(post_text, post_id):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": post_text}
            ],
            response_format={"type": "json_object"}
        )
        data = json.loads(response.choices[0].message.content)
        return data.get("sentences", [])
    except Exception as e:
        print(f"  ERROR {post_id}: {e}")
        return []

In [27]:
# ── Test on 3 posts before running the full pipeline ─────────────────────────
df = pd.read_csv(POSTS_PATH)
test_posts = df[["post_id", "selftext"]].dropna(subset=["selftext"]).head(3)

for _, row in test_posts.iterrows():
    sentences = extract_sentences(row["selftext"], row["post_id"])
    print(f"\n{'='*60}")
    print(f"POST: {row['post_id']}  ({len(sentences)} sentences extracted)")
    for s in sentences:
        print(f"  [{s['section']}]")
        if s['original'] != s['extracted']:
            print(f"    ORIGINAL:  {s['original']}")
            print(f"    EXTRACTED: {s['extracted']}")
        else:
            print(f"    {s['extracted']}")


POST: t3_1h2h9cn  (5 sentences extracted)
  [family_father]
    My dad was a truck driver and was never home and when he was he neglected me and my sister basically treating us like we weren't really there.
  [family_mother]
    Mom sadly was the same I had to see them get into a fist fight when I was a kid and the cops got called that prob did a lot of damage to me mentally if I had to guess.
  [bullying]
    I was bullied hard in school, I was beat and mocked for being white, short and wearing glasses so much so that I refused to wear glasses for years making it hard to do anything because my eyesight was so bad I developed insomnia giving me dark circles under my eyes to which people said I was a crack/meth head ...
  [self_harm]
    I started to self harm as a coping mechanism at first I would grind pencils into my flesh until it would become raw and bloody then when that stopped being enough I would pull my hair not out but grab a club and just pull for the pain, I would punch my

In [28]:
# ── Full extraction run ───────────────────────────────────────────────────────
df = pd.read_csv(POSTS_PATH)
posts = df[["post_id", "selftext"]].dropna(subset=["selftext"])

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        checkpoint = json.load(f)
    processed_ids = set(checkpoint["processed_ids"])
    all_sentences = checkpoint["sentences"]
    print(f"Resuming from checkpoint: {len(processed_ids)}/{len(posts)} posts done")
else:
    processed_ids = set()
    all_sentences = []
    print(f"Starting fresh: {len(posts)} posts to process")

for _, row in posts.iterrows():
    post_id = row["post_id"]
    if post_id in processed_ids:
        continue

    sentences = extract_sentences(row["selftext"], post_id)
    for s in sentences:
        s["post_id"] = post_id
        all_sentences.append(s)

    processed_ids.add(post_id)

    if len(processed_ids) % 50 == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump({"processed_ids": list(processed_ids), "sentences": all_sentences}, f)
        print(f"  [{len(processed_ids)}/{len(posts)}] checkpoint — {len(all_sentences)} sentences so far")

print(f"\nDone. {len(all_sentences)} sentences from {len(processed_ids)} posts.")

Starting fresh: 274 posts to process
  [50/274] checkpoint — 277 sentences so far
  [100/274] checkpoint — 542 sentences so far
  [150/274] checkpoint — 769 sentences so far
  [200/274] checkpoint — 1009 sentences so far
  [250/274] checkpoint — 1263 sentences so far

Done. 1391 sentences from 274 posts.


In [29]:
# ── Save sentence pool ────────────────────────────────────────────────────────
pool_df = pd.DataFrame(all_sentences)
pool_df = pool_df[["post_id", "section", "original", "extracted"]]
pool_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved → {OUTPUT_PATH}")
print(f"\nSentences per section:")
print(pool_df["section"].value_counts().to_string())
print(f"\nTotal: {len(pool_df)} sentences")

Saved → ../../data/sentence_pool.csv

Sentences per section:
section
suicidal_ideation    573
bullying             284
self_harm            231
family_mother        121
sa                   105
family_father         77

Total: 1391 sentences


In [30]:
# Dataset word count

pool_df["extracted"].apply(lambda text: len(str(text).split())).sum()

np.int64(26114)

# Stage 2: Section Composition

In [37]:
from hannah_profile import HANNAH_CANON_PROFILE


COMPOSE_SYSTEM_PROMPT = f"""\
You are composing one section of a first-person autobiography for Hannah.
You will be given a numbered list of sentences extracted from
real Reddit posts. You will also be given canon profile of hannah.
Your job is to extract the sentences into a coherent first-person narrative that reflects the canon profile provided.

RULES:
- Use only the sentences provided. Do not invent new content.
- You may add minimal connective tissue (2-3 words maximum) to join sentences that would
  otherwise read as completely disconnected — e.g. "and then", "after that", "i don't know".
  Nothing more.
- Preserve all original voice: spelling errors, lowercase, run-ons, fragments, bad punctuation.
- You must not make any changes to the sentences themselves.
- Do not summarise. Do not add reflection. Do not resolve the emotion.
- The output should read like Hannah wrote it unsupervised, and it should preserve the original tone / writing style of the sentences.
- Aim to use as many sentences as possible without compromising on continuity with other sentences.

{HANNAH_CANON_PROFILE}

Return only the composed section text. No preamble, no label, no explanation.
"""

In [38]:
def compose_section(section_name, sentences):
    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sentences))
    user_msg = f"Section: {section_name}\n\nSentences:\n{numbered}"
    try:
        response = client.chat.completions.create(
            model="gpt-5.4",
            messages=[
                {"role": "system", "content": COMPOSE_SYSTEM_PROMPT},
                {"role": "user", "content": user_msg}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"  ERROR composing {section_name}: {e}")
        return ""

In [39]:
# ── Compose all sections ──────────────────────────────────────────────────────
pool_df = pd.read_csv(OUTPUT_PATH)

SECTIONS = ["family_father", "family_mother", "sa", "bullying", "self_harm", "suicidal_ideation"]
AUTO_JSON_PATH = "../../data/hannah_autobiography.json"
AUTO_MD_PATH = "../../data/hannah_autobiography.md"

autobiography = {}

for section in SECTIONS:
    section_sentences = pool_df[pool_df["section"] == section]["extracted"].tolist()
    print(f"\n{'='*60}")
    print(f"{section.upper()}  ({len(section_sentences)} sentences)")
    print("="*60)

    if not section_sentences:
        print("  No sentences — skipping.")
        autobiography[section] = ""
        continue

    autobiography[section] = compose_section(section, section_sentences)
    print(autobiography[section])

# Save JSON
with open(AUTO_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(autobiography, f, indent=2, ensure_ascii=False)

# Save markdown
with open(AUTO_MD_PATH, "w", encoding="utf-8") as f:
    f.write("# Hannah — Autobiography\n\n")
    for section, text in autobiography.items():
        f.write(f"## {section.replace('_', ' ').title()}\n\n")
        f.write(text + "\n\n")

print(f"\nSaved → {AUTO_JSON_PATH}")
print(f"Saved → {AUTO_MD_PATH}")


FAMILY_FATHER  (77 sentences)
my parents split up when i was really young and i don’t recall them being together. My dad left me when i was young and i think part of it is because my mom didn’t let me see him when they broke up. My dad is out of the picture because he is an asshole bitch. My father abandoned us and stole everything when I was 9.

My father was verbally abusive and very scary at times. My dad was a truck driver and was never home and when he was he neglected me and my sister basically treating us like we weren't really there. My dad was present in the house but he would always close his door when I was home. My dad had kids at 16 so I assume he just never wanted to be a parent. I had nobody to run to at home either because my father was emotionally absent.

My father would hit me and yell at me, while my mom would just bystand. My dad would hit me with a belt when I was three years old. I'm sure my dad had good intentions deep down, but all I remember about him is how 